In [ ]:
import torch
import os
from video_processing import load_video_frames_opencv, extract_embedding
from transformers import TimesformerModel, TimesformerConfig
import faiss
import pickle
from tqdm import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
config = TimesformerConfig.from_pretrained("facebook/timesformer-base-finetuned-k400")
model = TimesformerModel.from_pretrained(
    "facebook/timesformer-base-finetuned-k400",
    trust_remote_code=True,
    use_safetensors=True)
model.eval().to(device)

In [ ]:
video_folder = "UCF101/train"
video_paths = []
for root, _, files in os.walk(video_folder):
    for f in files:
        if f.endswith((".mp4", ".avi")):
            video_paths.append(os.path.join(root, f))
len(video_paths)

In [ ]:
dimension = 768
index = faiss.IndexFlatL2(dimension)
embedding_map = {}
for path in tqdm(video_paths, desc="Processing videos"):
    try:
        frames = load_video_frames_opencv(path)
        emb = extract_embedding(frames, model, device)
        index.add(emb)
        embedding_map[len(embedding_map)] = path
    except Exception as e:
        print(f"Failed to process {path}: {e}")


In [ ]:
query_video = load_video_frames_opencv(r"UCF101\test\BalanceBeam\v_BalanceBeam_g05_c04.avi")
query_embedding = extract_embedding(query_video, model, device)
D, I = index.search(query_embedding, k=5)  

print("Top similar videos:")
for idx in I[0]:
    print(embedding_map[idx])


In [ ]:
print("Saving index and map...")
faiss.write_index(index, "embeddings/faiss_ucf101.index")
with open("embeddings/embedding_map.pkl", "wb") as f:
    pickle.dump(embedding_map, f)
print("Saved successfully.")